# 03 — Modelos de detección de anomalías · CERT r4.2

**Objetivos de este notebook (Fase 3 del PLAN.md):**
1. Entrenar un **baseline de reglas** (referencia interpretable tipo SIEM clásico).
2. Entrenar un **Isolation Forest** no supervisado sobre las 26 features z-score (`z_user_*`, `z_role_*`).
3. Entrenar un **autoencoder** (PyTorch) que aprende la normalidad y usa el error de reconstrucción como score de anomalía.
4. Producir un **score de riesgo por usuario-día** para cada modelo (convención: **mayor = más anómalo** en los tres, para que sean comparables) y guardarlo en `user_day_scores.parquet`.
5. Hacer una **evaluación compacta** contra el ground truth: AUC-PR a nivel usuario-día y a nivel usuario, y recall a presupuesto de investigación.

**Recordatorio importante:** los tres modelos son **no supervisados** — entrenan únicamente sobre las 26 columnas `z_user_*`/`z_role_*`. Las etiquetas (`label_malicious_day`, `is_insider_user`, `scenario`) se usan **solo para evaluar** al final, nunca como input.

In [ ]:
# [compute]
# === Setup: entorno, dependencias y rutas ===
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    %pip install -q polars pyarrow
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/CERT_data")
    DATA_DIR = BASE / "data"/ "r4.2"
    ANSWERS_DIR = BASE / "answers"
    PROCESSED_DIR = BASE / "processed"
else:
    # Local: el notebook vive en SIEM/notebooks/ — reutilizamos src/config.py
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    sys.path.insert(0, str(PROJECT_ROOT))
    from src.config import DATA_DIR, ANSWERS_DIR, PROCESSED_DIR

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

import polars as pl

DATE_FORMAT = "%m/%d/%Y %H:%M:%S"
WORK_HOURS = (8, 18)

print(f"Entorno: {'Colab' if IN_COLAB else 'Local'}")
print(f"Datos:   {DATA_DIR}")
print(f"Answers: {ANSWERS_DIR}")
print(f"Procesados: {PROCESSED_DIR}")

# --- Dependencias de modelado: instalación condicional si faltan en local ---
try:
    import numpy as np
    from sklearn.ensemble import IsolationForest
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import average_precision_score, roc_auc_score
    import torch
except ImportError:
    import subprocess, sys as _sys
    subprocess.run([_sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "torch", "numpy"], check=True)
    import numpy as np
    from sklearn.ensemble import IsolationForest
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import average_precision_score, roc_auc_score
    import torch

In [ ]:
# [compute]
# === Carga de datos y construcción de la matriz de modelo ===

df = pl.read_parquet(PROCESSED_DIR / "user_day_features.parquet")

model_features = [c for c in df.columns if c.startswith("z_user_") or c.startswith("z_role_")]
assert len(model_features) == 26, f"esperadas 26 features z-score, obtenidas {len(model_features)}"

X = df.select(model_features).to_numpy()

# Estandarización global de la matriz (los z-scores ya están normalizados por
# grupo, pero estandarizamos la matriz completa para que IF y el autoencoder
# trabajen sobre una escala homogénea entre las 26 columnas).
scaler = StandardScaler()
Xs = scaler.fit_transform(X)

y_day = df["label_malicious_day"].to_numpy()

print(f"df:    {df.shape}")
print(f"X:     {X.shape}")
print(f"Xs:    {Xs.shape}")
print(f"y_day: {y_day.shape}, positivos: {y_day.sum()} ({y_day.mean():.4%})")
print(f"nº model_features: {len(model_features)}")

## Baseline de reglas

El SIEM clásico se basa en reglas/umbrales sobre indicadores conocidos. Aquí construimos un
score interpretable: para cada usuario-día, sumamos **"cuántas desviaciones por encima de su
propia normalidad"** presenta en un conjunto de comportamientos sospechosos (USB fuera de
horario, logons fuera de horario, logons en PCs ajenos, emails externos, copias de fichero...).

Solo se cuenta la parte **positiva** del `z_user_*` (un valor por debajo de la media no es
sospechoso) y se suman las desviaciones de las distintas señales. Es un baseline simple e
interpretable — sirve de referencia para justificar (o no) el uso de modelos de ML más complejos.

In [ ]:
# [compute]
# === Baseline de reglas: suma de desviaciones positivas en señales sospechosas ===

suspicious = [
    "z_user_n_afterhours_usb",
    "z_user_n_usb_connects",
    "z_user_n_afterhours_logons",
    "z_user_n_other_pc_logons",
    "z_user_n_external_emails",
    "z_user_n_file_copies",
    "z_user_n_afterhours_file_copies",
]

score_rules = df.select(suspicious).to_numpy().clip(min=0).sum(axis=1)

print(f"score_rules: min={score_rules.min():.3f}, max={score_rules.max():.3f}, mean={score_rules.mean():.3f}")

## Isolation Forest

Modelo de detección de anomalías por **aislamiento**: construye árboles aleatorios que parten
el espacio de features, y las observaciones anómalas (las 26 columnas z-score) requieren menos
particiones para quedar aisladas que las normales. No supervisado: no usa las etiquetas,
solo la estructura de la matriz `Xs`.

In [ ]:
# [compute]
# === Isolation Forest ===

iforest = IsolationForest(n_estimators=200, contamination="auto", random_state=42, n_jobs=-1)
iforest.fit(Xs)

# score_samples: mayor = más normal -> negamos para que mayor = más anómalo
score_iforest = -iforest.score_samples(Xs)

print(f"score_iforest: min={score_iforest.min():.4f}, max={score_iforest.max():.4f}, mean={score_iforest.mean():.4f}")

## Autoencoder (PyTorch)

Un autoencoder se entrena para **reconstruir** su entrada a través de un cuello de botella
(26 → 16 → 8 → 16 → 26). Como la inmensa mayoría de los usuario-día son comportamiento
benigno, el modelo aprende a representar bien la **normalidad** y comete **mayor error de
reconstrucción** en los días anómalos (que apenas ha visto durante el entrenamiento). El
error de reconstrucción (MSE por fila) es, por tanto, el score de anomalía.

In [ ]:
# [compute]
# === Autoencoder en PyTorch (CPU, determinista) ===

torch.manual_seed(42)
np.random.seed(42)

n_features = Xs.shape[1]

autoencoder = torch.nn.Sequential(
    torch.nn.Linear(n_features, 16),
    torch.nn.ReLU(),
    torch.nn.Linear(16, 8),
    torch.nn.ReLU(),
    torch.nn.Linear(8, 16),
    torch.nn.ReLU(),
    torch.nn.Linear(16, n_features),
)

criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)

Xt = torch.tensor(Xs, dtype=torch.float32)
dataset = torch.utils.data.TensorDataset(Xt)
loader = torch.utils.data.DataLoader(dataset, batch_size=512, shuffle=True)

EPOCHS = 30
autoencoder.train()
for epoch in range(1, EPOCHS + 1):
    total_loss = 0.0
    n_batches = 0
    for (batch,) in loader:
        optimizer.zero_grad()
        recon_batch = autoencoder(batch)
        loss = criterion(recon_batch, batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    if epoch % 5 == 0:
        print(f"epoch {epoch:2d}/{EPOCHS} - loss medio: {total_loss / n_batches:.5f}")

autoencoder.eval()
with torch.no_grad():
    recon = autoencoder(Xt)
    score_autoencoder = ((recon - Xt) ** 2).mean(dim=1).numpy()

print(f"score_autoencoder: min={score_autoencoder.min():.4f}, max={score_autoencoder.max():.4f}, mean={score_autoencoder.mean():.4f}")

In [ ]:
# [compute]
# === Ensamblar scores y guardar parquet ===

scores = df.select("user", "day", "label_malicious_day", "is_insider_user", "scenario").with_columns(
    pl.Series("score_rules", score_rules),
    pl.Series("score_iforest", score_iforest),
    pl.Series("score_autoencoder", score_autoencoder),
)

out_path = PROCESSED_DIR / "user_day_scores.parquet"
scores.write_parquet(out_path)

print(f"Guardado en: {out_path}")
print(f"scores: {scores.shape}")
scores.head(5)

## Evaluación compacta

Dado el **desbalanceo extremo** (>99.9% de los usuario-día son benignos), **accuracy no aporta
información**: un modelo que prediga "todo benigno" tendría accuracy >99.9% y recall 0.
La métrica clave es **AUC-PR (average precision)**, que resume el trade-off precision/recall
en todos los umbrales y no se infla con la clase mayoritaria.

Evaluamos en dos niveles:
- **Usuario-día**: ¿el modelo asigna scores altos a los días concretos en que ocurre el ataque?
- **Usuario**: agregando el score por usuario (máximo sobre sus días), ¿el modelo identifica
  a los 70 usuarios insider del dataset?

Y una métrica **operativa de SOC**: **recall a presupuesto de investigación** — si un analista
solo puede investigar el 1% o el 5% de los días-usuario más anómalos (ordenados por score),
¿qué fracción de los días maliciosos reales caería dentro de ese presupuesto?

In [ ]:
# [compute]
# === Evaluación compacta: AUC-PR/AUC-ROC, recall a presupuesto, nivel usuario ===

model_cols = ["score_rules", "score_iforest", "score_autoencoder"]
budgets = [0.01, 0.05]
N = len(y_day)

scores_pd = scores.to_pandas()

rows = []
for m in model_cols:
    score = scores_pd[m].to_numpy()

    # --- Nivel usuario-día ---
    ap_day = average_precision_score(y_day, score)
    auc_day = roc_auc_score(y_day, score)

    # --- Recall a presupuesto de investigación ---
    order = np.argsort(-score)
    y_sorted = y_day[order]
    total_positives = y_day.sum()
    recall_budgets = {}
    for budget in budgets:
        k = int(budget * N)
        recall_budgets[budget] = y_sorted[:k].sum() / total_positives

    # --- Nivel usuario (agregación por máximo) ---
    user_agg = (
        scores.group_by("user")
        .agg(
            pl.col(m).max().alias("user_score"),
            pl.col("is_insider_user").any().alias("y_user"),
        )
    )
    user_score = user_agg["user_score"].to_numpy()
    y_user = user_agg["y_user"].to_numpy().astype(int)

    ap_user = average_precision_score(y_user, user_score)

    n_insiders = int(y_user.sum())
    top_idx = np.argsort(-user_score)[:n_insiders]
    det_top70 = y_user[top_idx].sum() / n_insiders

    rows.append({
        "modelo": m,
        "ap_day": ap_day,
        "auc_day": auc_day,
        "recall@1%": recall_budgets[0.01],
        "recall@5%": recall_budgets[0.05],
        "ap_user": ap_user,
        "det_top70": det_top70,
    })

    assert ap_day > 0.01, f"{m}: ap_day={ap_day:.4f} demasiado bajo (azar ~{y_day.mean():.4f})"
    assert auc_day > 0.5, f"{m}: auc_day={auc_day:.4f} no mejor que azar"

results = pl.DataFrame(rows)

print(results)
results

## Conclusiones

_(plantilla a rellenar tras revisar la tabla de resultados anterior)_

- **Modelo con mejor AUC-PR a nivel usuario-día**: _completar_ — indica qué tan bien
  el score señala los días concretos de actividad maliciosa.
- **Modelo con mejor detección a nivel usuario (`det_top70`)**: _completar_ — relevante para
  priorizar la watchlist de analistas.
- **Trade-off recall vs alertas**: el `recall@1%`/`recall@5%` muestra cuántos días maliciosos
  se capturarían investigando solo el 1%/5% de los días más anómalos — comparar este coste
  operativo entre modelos (más recall a igual presupuesto = mejor priorización del SOC).
- **Baseline de reglas vs ML**: discutir si Isolation Forest y/o el autoencoder superan al
  baseline interpretable, y en qué medida justifica su mayor complejidad.

Los scores de los tres modelos quedan guardados en `data/processed/user_day_scores.parquet`
(columnas `score_rules`, `score_iforest`, `score_autoencoder`, junto con las etiquetas de
evaluación). La **fase 4** hará la comparativa visual completa (curvas PR, alertas/día por
umbral, distribución de scores) y la **fase 5** integrará estos scores en el dashboard SOC.